# Fase 2 — Treino do modelo de classificação

**Modelo:** EfficientNetB0 pré-treinado no ImageNet, com *transfer learning* em duas fases (secção 4.6.3 do relatório).

- **Fase 1:** base congelada; treina-se apenas a cabeça de classificação (Adam, lr = 1e-3).
- **Fase 2 (*fine-tuning*):** descongelam-se as últimas ~20 camadas da base, mantendo as BatchNormalization congeladas (Adam, lr = 1e-5).

Treina-se **um modelo por cultura**. Corra o notebook uma vez com `CULTURA = "milho"` e outra com `CULTURA = "mandioca"`.

**Antes de começar (Google Colab):**
1. Menu *Ambiente de execução → Alterar tipo de ambiente de execução*: **GPU T4** e uma versão do ambiente com **Python 3.12** (ex.: 2025.10). As versões mais recentes do Colab trazem Python 3.13, que o TensorFlow 2.17.1 não suporta.
2. No Google Drive deve existir `O meu disco/PFC/` com `ccmt_raw.zip` e a pasta `divisoes/`.

O conjunto de **teste não é usado aqui**: só é avaliado na Fase 3, para que o resultado final seja uma estimativa honesta.

## 1. Versões das bibliotecas

In [ ]:
# Instala no Colab as mesmas versões usadas no PC (ml/requirements.txt), para que
# o modelo treinado abra sem problemas na avaliação (Fase 3) e na exportação (Fase 4).
# Na primeira execução a sessão reinicia sozinha no fim: depois disso, volte a
# correr o notebook desde o início (esta célula passa à frente).
import importlib.metadata as md
import os
import re
import shutil
import sys

if sys.version_info >= (3, 13):
    raise RuntimeError(
        f"Este ambiente tem Python {sys.version.split()[0]}, mas o TensorFlow 2.17.1 só "
        "suporta até 3.12. No Colab: Ambiente de execução → Alterar tipo de ambiente de "
        "execução → em 'Versão do ambiente de execução' escolha uma versão anterior "
        "(ex.: 2025.10, com Python 3.12), mantendo a GPU T4.")

VERSOES = {"tensorflow": "2.17.1", "keras": "3.15.1", "numpy": "1.26.4",
           "scikit-learn": "1.5.2", "pandas": "2.2.3"}
TEM_GPU = shutil.which("nvidia-smi") is not None


def versao_instalada(pacote):
    try:
        return md.version(pacote)
    except md.PackageNotFoundError:
        return None


def bibliotecas_cuda_em_falta():
    # Em Linux, o TensorFlow só usa a GPU com as bibliotecas CUDA exactas que
    # declara no extra "and-cuda" (ex.: nvidia-cudnn-cu12==8.9.7.29).
    # Formato dos requisitos: "nvidia-cublas-cu12 (==12.3.4.1) ; extra == 'and-cuda'"
    em_falta = []
    for requisito in md.requires("tensorflow") or []:
        r = re.match(r"([\w.\-]+)\s*\(?==([^);\s]+)\)?\s*;.*and-cuda", requisito)
        if r and versao_instalada(r.group(1)) != r.group(2):
            em_falta.append(r.group(1))
    return em_falta


em_falta = {p: v for p, v in VERSOES.items() if versao_instalada(p) != v}
if TEM_GPU and not em_falta and bibliotecas_cuda_em_falta():
    em_falta["tensorflow"] = VERSOES["tensorflow"]

if em_falta:
    pacotes = [f"{p}=={v}" for p, v in em_falta.items()]
    if TEM_GPU and "tensorflow" in em_falta:
        pacotes = [f"tensorflow[and-cuda]=={VERSOES['tensorflow']}" if p.startswith("tensorflow==")
                   else p for p in pacotes]
    pacotes = " ".join(f'"{p}"' for p in pacotes)
    print("A instalar:", pacotes)
    !pip install -q {pacotes}
    print("Instalação concluída. A sessão vai reiniciar: corra o notebook outra vez desde o início.")
    os.kill(os.getpid(), 9)
else:
    print("Versões correctas:", VERSOES, "| GPU:", "sim" if TEM_GPU else "não")

## 2. Configuração da experiência

In [ ]:
CULTURA = "milho"  # mudar para "mandioca" para treinar o segundo modelo

# Composição do conjunto de treino: variável independente da investigação (secção 4.7).
# Validação e teste usam SEMPRE todas as fontes, para que as experiências sejam comparáveis.
EXPERIENCIA = "ccmt"       # nome da experiência (pasta onde ficam os resultados)
FONTES_TREINO = ["ccmt"]   # ex.: ["ccmt", "tecnicos", "campo"] quando houver imagens locais

RANDOM_STATE = 42          # o mesmo usado na divisão dos dados (ml/src/config.py)
TAMANHO_IMAGEM = 224       # resolução de entrada nativa do EfficientNetB0
TAMANHO_LOTE = 32

EPOCAS_FASE1 = 20          # máximos; o EarlyStopping pode parar antes
EPOCAS_FASE2 = 30
PACIENCIA = 5              # épocas sem melhoria de val_loss até parar

LR_FASE1 = 1e-3
LR_FASE2 = 1e-5            # 100 vezes menor (secção 4.6.3); experiência: 1e-4 ou 5e-5
DROPOUT = 0.3

# Que camadas da base descongelar na Fase 2:
#   "ultimas_20"                 -> últimas 20 camadas (configuração base)
#   ("block6", "block7", "top")  -> blocos inteiros (ajuste fino mais profundo)
# As BatchNormalization ficam sempre congeladas.
DESCONGELAR = "ultimas_20"
LABEL_SMOOTHING = 0.0      # experiência: 0.1 (ajuda com classes visualmente próximas)

## 3. Bibliotecas, GPU e semente aleatória

In [ ]:
import json
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU activa:", gpus[0].name)
else:
    print("ATENÇÃO: sem GPU. No Colab: Ambiente de execução → Alterar tipo de "
          "ambiente de execução → GPU T4. Sem GPU o treino demora muitas horas.")

# Fixa a semente do Python, do NumPy e do TensorFlow. Na GPU alguns cálculos
# não são totalmente determinísticos, por isso podem surgir pequenas diferenças
# entre execuções; a divisão dos dados, essa, é sempre a mesma (vem do CSV).
keras.utils.set_random_seed(RANDOM_STATE)

## 4. Ligação aos dados

In [ ]:
EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PFC = Path("/content/drive/MyDrive/PFC")
    PASTA_RAW = Path("/content/raw")  # disco local do Colab: muito mais rápido que o Drive
    if not PASTA_RAW.exists():
        print("A descomprimir ccmt_raw.zip (alguns minutos)...")
        shutil.unpack_archive(str(PASTA_PFC / "ccmt_raw.zip"), str(PASTA_RAW))
    PASTA_DIVISOES = PASTA_PFC / "divisoes"
    PASTA_SAIDA = PASTA_PFC / "resultados" / CULTURA / EXPERIENCIA
else:
    # Execução local no PC, com o notebook aberto a partir de ml/notebooks/.
    PASTA_ML = Path.cwd().parent
    PASTA_RAW = PASTA_ML / "data" / "raw"
    PASTA_DIVISOES = PASTA_ML / "data" / "divisoes"
    PASTA_SAIDA = PASTA_ML / "models" / CULTURA / EXPERIENCIA

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
print("Imagens:   ", PASTA_RAW)
print("Divisão:   ", PASTA_DIVISOES / f"divisao_{CULTURA}.csv")
print("Resultados:", PASTA_SAIDA)

## 5. Leitura da divisão treino/validação/teste

A divisão vem do CSV gerado por `ml/src/dividir_dados.py` e é a mesma em todas as experiências.

In [ ]:
divisao = pd.read_csv(PASTA_DIVISOES / f"divisao_{CULTURA}.csv")

# Ordem alfabética: o índice de cada classe fica fixo e é guardado com o modelo,
# porque a API e a app precisam dele para traduzir a saída do modelo.
CLASSES = sorted(divisao["classe"].unique())
INDICE = {classe: i for i, classe in enumerate(CLASSES)}

divisao["rotulo"] = divisao["classe"].map(INDICE)
divisao["ficheiro"] = [str(PASTA_RAW / caminho) for caminho in divisao["caminho"]]

em_falta = [f for f in divisao["ficheiro"] if not Path(f).exists()]
assert not em_falta, f"{len(em_falta)} imagens do CSV não existem, ex.: {em_falta[:3]}"

treino_df = divisao[(divisao["subconjunto"] == "treino") & divisao["fonte"].isin(FONTES_TREINO)]
validacao_df = divisao[divisao["subconjunto"] == "validacao"]
assert set(treino_df["classe"]) == set(CLASSES), "Há classes sem imagens de treino nas fontes escolhidas"

print("Imagens de treino por fonte:", treino_df["fonte"].value_counts().to_dict())

print(f"Classes ({len(CLASSES)}):", CLASSES)
pd.crosstab(divisao["classe"], divisao["subconjunto"], margins=True, margins_name="total")

## 6. Pipeline de imagens e aumento de dados

- As imagens são redimensionadas para 224×224 e **mantidas na escala 0–255**: o EfficientNetB0 do Keras já inclui a normalização como camada interna, por isso **não se divide por 255**.
- O aumento de dados (regra 4 / secção 2.3.3.2) usa camadas Keras e é aplicado **só ao treino**, no pipeline de dados. Assim o modelo guardado não contém camadas aleatórias, o que simplifica a conversão para TFLite.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
N_CLASSES = len(CLASSES)

aumento_de_dados = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(30 / 360),                  # até ±30 graus
    layers.RandomZoom(0.2),                           # zoom entre 0,8 e 1,2
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2),
], name="aumento_de_dados")


def ler_imagem(caminho, rotulo):
    imagem = tf.io.decode_image(tf.io.read_file(caminho), channels=3, expand_animations=False)
    imagem = tf.image.resize(imagem, (TAMANHO_IMAGEM, TAMANHO_IMAGEM))
    # Guardada em memória como uint8 (4x menos RAM que float32); valores 0-255.
    imagem = tf.cast(tf.clip_by_value(tf.round(imagem), 0, 255), tf.uint8)
    return imagem, tf.one_hot(rotulo, N_CLASSES)


def criar_dataset(df, treino):
    ds = tf.data.Dataset.from_tensor_slices((df["ficheiro"].values, df["rotulo"].values))
    ds = ds.map(ler_imagem, num_parallel_calls=AUTOTUNE).cache()
    if treino:
        ds = ds.shuffle(len(df), seed=RANDOM_STATE, reshuffle_each_iteration=True)
    ds = ds.batch(TAMANHO_LOTE)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if treino:
        ds = ds.map(lambda x, y: (aumento_de_dados(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


treino_ds = criar_dataset(treino_df, treino=True)
validacao_ds = criar_dataset(validacao_df, treino=False)

In [ ]:
# Pré-visualização do aumento de dados: a mesma imagem de treino, 8 vezes.
exemplo, _ = ler_imagem(treino_df["ficheiro"].iloc[0], treino_df["rotulo"].iloc[0])
exemplo = tf.cast(exemplo, tf.float32)[tf.newaxis]

fig, eixos = plt.subplots(2, 4, figsize=(12, 6))
for i, eixo in enumerate(eixos.flat):
    imagem = exemplo if i == 0 else aumento_de_dados(exemplo, training=True)
    eixo.imshow(tf.cast(imagem[0], tf.uint8))
    eixo.set_title("original" if i == 0 else f"aumentada {i}")
    eixo.axis("off")
plt.tight_layout()
plt.savefig(PASTA_SAIDA / "exemplos_aumento_de_dados.png", dpi=120)
plt.show()

## 7. Pesos por classe

Algumas classes têm menos imagens (secção 4.6.7). Com `class_weight="balanced"`, os erros nessas classes pesam mais na função de perda, para que o modelo não as ignore.

In [ ]:
pesos = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=treino_df["rotulo"])
PESOS_CLASSES = dict(enumerate(pesos))

pd.DataFrame({
    "classe": CLASSES,
    "imagens_treino": treino_df["rotulo"].value_counts().sort_index().values,
    "peso": np.round(pesos, 3),
})

## 8. Construção do modelo

`base(entradas, training=False)` mantém as camadas BatchNormalization em modo de inferência, mesmo depois de descongeladas na Fase 2 — é a prática recomendada pelo Keras para *fine-tuning*, porque evita estragar as estatísticas aprendidas no ImageNet.

In [ ]:
base = keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(TAMANHO_IMAGEM, TAMANHO_IMAGEM, 3),
)
base.trainable = False  # Fase 1: base congelada

entradas = keras.Input(shape=(TAMANHO_IMAGEM, TAMANHO_IMAGEM, 3), name="imagem")
x = base(entradas, training=False)
x = layers.GlobalAveragePooling2D(name="pooling")(x)
x = layers.Dropout(DROPOUT, name="dropout")(x)
saidas = layers.Dense(N_CLASSES, activation="softmax", name="classificacao")(x)
modelo = keras.Model(entradas, saidas, name=f"efficientnetb0_{CULTURA}")

modelo.summary()

In [ ]:
def criar_callbacks(fase):
    return [
        # Pára quando val_loss deixa de melhorar e repõe os melhores pesos.
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PACIENCIA,
                                      restore_best_weights=True, verbose=1),
        # Guarda no Drive o melhor modelo da fase (protege contra quedas do Colab).
        keras.callbacks.ModelCheckpoint(str(PASTA_SAIDA / f"melhor_{fase}.keras"),
                                        monitor="val_loss", save_best_only=True, verbose=1),
        keras.callbacks.CSVLogger(str(PASTA_SAIDA / f"historial_{fase}.csv")),
    ]

## 9. Fase 1 — treino da cabeça de classificação (base congelada)

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FASE1),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

inicio = time.time()
historial_1 = modelo.fit(
    treino_ds, validation_data=validacao_ds, epochs=EPOCAS_FASE1,
    class_weight=PESOS_CLASSES, callbacks=criar_callbacks("fase1"),
)
duracao_fase1 = time.time() - inicio
print(f"Fase 1 concluída em {duracao_fase1 / 60:.1f} min")

## 10. Fase 2 — *fine-tuning* das últimas camadas

In [ ]:
base.trainable = True
ultimas_20 = set(c.name for c in base.layers[-20:])
for camada in base.layers:
    if DESCONGELAR == "ultimas_20":
        treinavel = camada.name in ultimas_20
    else:
        treinavel = camada.name.startswith(tuple(DESCONGELAR))
    # BatchNormalization sempre congeladas
    camada.trainable = treinavel and not isinstance(camada, layers.BatchNormalization)

treinaveis = [c.name for c in base.layers if c.trainable]
print(f"Camadas da base descongeladas: {len(treinaveis)} de {len(base.layers)}")
print(f"Parâmetros treináveis: {sum(int(np.prod(w.shape)) for w in modelo.trainable_weights):,}")

# É obrigatório recompilar depois de mudar o que é treinável.
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FASE2),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

epocas_feitas = len(historial_1.epoch)
inicio = time.time()
historial_2 = modelo.fit(
    treino_ds, validation_data=validacao_ds,
    initial_epoch=epocas_feitas, epochs=epocas_feitas + EPOCAS_FASE2,
    class_weight=PESOS_CLASSES, callbacks=criar_callbacks("fase2"),
)
duracao_fase2 = time.time() - inicio
print(f"Fase 2 concluída em {duracao_fase2 / 60:.1f} min")

## 11. Curvas de aprendizagem

In [ ]:
h = {chave: historial_1.history[chave] + historial_2.history[chave]
     for chave in ["accuracy", "val_accuracy", "loss", "val_loss"]}
epocas = range(1, len(h["loss"]) + 1)

fig, (eixo_acc, eixo_perda) = plt.subplots(1, 2, figsize=(13, 4.5))
for eixo, metrica, titulo in [(eixo_acc, "accuracy", "Acurácia"),
                              (eixo_perda, "loss", "Perda (entropia cruzada)")]:
    eixo.plot(epocas, h[metrica], label="treino")
    eixo.plot(epocas, h[f"val_{metrica}"], label="validação")
    eixo.axvline(epocas_feitas + 0.5, color="grey", linestyle="--", label="início da fase 2")
    eixo.set_title(f"{titulo} — {CULTURA}")
    eixo.set_xlabel("Época")
    eixo.grid(alpha=0.3)
    eixo.legend()
plt.tight_layout()
plt.savefig(PASTA_SAIDA / "curvas_treino.png", dpi=150)
plt.show()

## 12. Guardar o modelo final

Guarda-se o modelo, a lista de classes (pela ordem das saídas do modelo) e um registo da configuração, para a experiência poder ser descrita e repetida no relatório.

In [ ]:
perda_val, acuracia_val = modelo.evaluate(validacao_ds, verbose=0)
print(f"Validação: acurácia = {acuracia_val:.4f} | perda = {perda_val:.4f}")
print("(O desempenho final é medido só no conjunto de teste, na Fase 3.)")

modelo.save(PASTA_SAIDA / f"modelo_{CULTURA}.keras")

with open(PASTA_SAIDA / f"classes_{CULTURA}.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False, indent=2)

registo = {
    "cultura": CULTURA,
    "experiencia": EXPERIENCIA,
    "fontes_treino": FONTES_TREINO,
    "imagens_treino_por_fonte": treino_df["fonte"].value_counts().to_dict(),
    "classes": CLASSES,
    "imagens": {"treino": len(treino_df), "validacao": len(validacao_df)},
    "hiperparametros": {
        "tamanho_imagem": TAMANHO_IMAGEM, "tamanho_lote": TAMANHO_LOTE,
        "epocas_max_fase1": EPOCAS_FASE1, "epocas_max_fase2": EPOCAS_FASE2,
        "paciencia": PACIENCIA, "lr_fase1": LR_FASE1, "lr_fase2": LR_FASE2,
        "descongelar": DESCONGELAR if isinstance(DESCONGELAR, str) else list(DESCONGELAR),
        "camadas_descongeladas": len(treinaveis), "label_smoothing": LABEL_SMOOTHING,
        "dropout": DROPOUT,
        "class_weight": "balanced", "random_state": RANDOM_STATE,
    },
    "epocas_treinadas": {"fase1": len(historial_1.epoch), "fase2": len(historial_2.epoch)},
    "duracao_min": {"fase1": round(duracao_fase1 / 60, 1), "fase2": round(duracao_fase2 / 60, 1)},
    "validacao": {"acuracia": round(acuracia_val, 4), "perda": round(perda_val, 4)},
    "versoes": {"tensorflow": tf.__version__, "keras": keras.__version__},
    "gpu": gpus[0].name if gpus else None,
}
with open(PASTA_SAIDA / f"registo_treino_{CULTURA}.json", "w", encoding="utf-8") as f:
    json.dump(registo, f, ensure_ascii=False, indent=2)

print("Ficheiros guardados em", PASTA_SAIDA)
for ficheiro in sorted(PASTA_SAIDA.iterdir()):
    print("  ", ficheiro.name)

## 13. Onde é que o modelo erra? (conjunto de **validação**)

Métricas por classe e matriz de confusão calculadas **na validação**, para orientar as experiências seguintes. O conjunto de teste continua intocado: só é usado uma vez, na Fase 3, com o modelo escolhido. Se o teste servisse para afinar o modelo, os resultados finais deixariam de ser uma estimativa honesta.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

probabilidades = modelo.predict(validacao_ds, verbose=0)
y_previsto = probabilidades.argmax(axis=1)
y_real = validacao_df["rotulo"].values  # mesma ordem: a validação não é baralhada

relatorio = pd.DataFrame(classification_report(
    y_real, y_previsto, labels=range(N_CLASSES), target_names=CLASSES,
    output_dict=True, zero_division=0)).T
relatorio.to_csv(PASTA_SAIDA / "validacao_metricas_por_classe.csv")

por_classe = relatorio.loc[CLASSES, ["precision", "recall", "f1-score", "support"]]
por_classe.columns = ["precisão", "sensibilidade", "F1", "imagens"]
abaixo = por_classe.index[por_classe["F1"] < 0.85].tolist()
print("Classes com F1 < 0,85 na validação:", abaixo or "nenhuma")
por_classe.round(3)

In [ ]:
# Matriz de confusão: cor = proporção da classe real (linha), texto = n.º de imagens.
matriz = confusion_matrix(y_real, y_previsto, labels=range(N_CLASSES))
proporcao = matriz / matriz.sum(axis=1, keepdims=True)
ACENTOS = {"saudavel": "saudável", "acaro_verde": "ácaro verde"}  # nomes para as figuras
nomes = [ACENTOS.get(c, c.replace("_", " ")) for c in CLASSES]

fig, eixo = plt.subplots(figsize=(1.1 * N_CLASSES + 3, 1.0 * N_CLASSES + 2))
imagem = eixo.imshow(proporcao, cmap="Blues", vmin=0, vmax=1)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if matriz[i, j]:
            eixo.text(j, i, matriz[i, j], ha="center", va="center", fontsize=10,
                      color="white" if proporcao[i, j] > 0.5 else "#1f2937")
eixo.set_xticks(range(N_CLASSES), nomes, rotation=40, ha="right")
eixo.set_yticks(range(N_CLASSES), nomes)
eixo.set_xlabel("Classe prevista")
eixo.set_ylabel("Classe real")
eixo.set_title(f"Matriz de confusão (validação) — {CULTURA}, experiência {EXPERIENCIA}")
for lado in eixo.spines.values():
    lado.set_visible(False)
fig.colorbar(imagem, ax=eixo, fraction=0.046, pad=0.04, label="Proporção da classe real")
plt.tight_layout()
plt.savefig(PASTA_SAIDA / "validacao_matriz_confusao.png", dpi=200)
plt.show()

# Pares de classes mais confundidos
pares = [(matriz[i, j], CLASSES[i], CLASSES[j])
         for i in range(N_CLASSES) for j in range(N_CLASSES) if i != j and matriz[i, j]]
for n, real, previsto in sorted(pares, reverse=True)[:5]:
    print(f"{n:4d} imagens de '{real}' classificadas como '{previsto}'")

## 14. Tabela de experiências

Cada execução acrescenta uma linha a `resultados/<cultura>/experiencias.csv`. Esta tabela relaciona as variáveis independentes (secção 4.7) com o desempenho na validação e é a base do Capítulo 5.

In [ ]:
linha = {
    "experiencia": EXPERIENCIA,
    "data": time.strftime("%Y-%m-%d %H:%M"),
    "fontes_treino": "+".join(FONTES_TREINO),
    "descongelar": DESCONGELAR if isinstance(DESCONGELAR, str) else "+".join(DESCONGELAR),
    "camadas_descongeladas": len(treinaveis),
    "lr_fase2": LR_FASE2,
    "label_smoothing": LABEL_SMOOTHING,
    "epocas": len(historial_1.epoch) + len(historial_2.epoch),
    "duracao_min": round((duracao_fase1 + duracao_fase2) / 60, 1),
    "val_acuracia": round(acuracia_val, 4),
    "val_f1_ponderado": round(relatorio.loc["weighted avg", "f1-score"], 4),
    "val_f1_minimo": round(por_classe["F1"].min(), 4),
    "classe_f1_minimo": por_classe["F1"].idxmin(),
}
tabela = PASTA_SAIDA.parent / "experiencias.csv"
experiencias = pd.concat([pd.read_csv(tabela), pd.DataFrame([linha])]) if tabela.exists() \
    else pd.DataFrame([linha])
experiencias.to_csv(tabela, index=False)
experiencias